# eThekwini institution inventory audit

Select the existing South African OSM records using the accepted municipal polygon. Retain all selected records, assign review flags, and export a local inventory. No independent coordinate verification, duplicate merging, flood-exposure analysis or publication is performed.

Requires the research environment and editable package installation from the README. The registered boundary files must be available locally; use notebook 00 if needed. Outputs are replaced on rerun; each execution records a new provenance run. Preserve human review decisions separately using source record IDs.

In [ ]:
from pathlib import Path
import json

# In Colab, clone the repository and change into it before running this cell.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout or one of its subdirectories.")
CONFIG = json.loads((ROOT / "config/ethekwini.yaml").read_text())
print("Project:", CONFIG["project_id"])


In [ ]:
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import platform
import subprocess
import geopandas as gpd
import pandas as pd
import shapely
import pyproj
from heritage_flood.inventory import audit_inventory

started = datetime.now(timezone.utc)
run_id = "inventory_audit_" + started.strftime("%Y%m%dT%H%M%S%fZ")
source_path = ROOT / CONFIG["institution_source"]
boundary_path = ROOT / CONFIG["boundary_path"]
collection = json.loads(source_path.read_text())
boundary = gpd.read_file(boundary_path)
# Verify that the configured boundary is the registered snapshot before use.
with (ROOT / "metadata/datasets.csv").open(newline="") as handle:
    registered = {r["dataset_id"]: r for r in csv.DictReader(handle)}
expected_boundary_hash = registered[CONFIG["boundary_dataset_id"]]["sha256"]
assert hashlib.sha256(boundary_path.read_bytes()).hexdigest() == expected_boundary_hash
result = audit_inventory(collection, boundary, analysis_crs=CONFIG["analysis_crs"],
                         **CONFIG["inventory_audit"])
institutions = result["institutions"]
institutions["institution_source_dataset_id"] = CONFIG["institution_dataset_id"]
institutions["boundary_dataset_id"] = CONFIG["boundary_dataset_id"]
institutions["run_id"] = run_id
print("Selection:", dict(Counter(result["selection"].selection_status)))
print("Selected records:", len(institutions))
print("Missing names:", int(institutions.missing_name.sum()))
print("Candidate duplicate/co-location pairs:", len(result["duplicate_candidates"]))


In [ ]:
output = ROOT / CONFIG["output_directory"]
output.mkdir(parents=True, exist_ok=True)
institutions.to_file(output / "institutions.geojson", driver="GeoJSON")
institutions.to_crs(CONFIG["analysis_crs"]).to_file(
    output / "institutions.gpkg", layer="institutions", driver="GPKG")
institutions.drop(columns="geometry").to_csv(output / "institutions.csv", index=False)
result["selection"].to_csv(output / "institution_selection.csv", index=False)
result["duplicate_candidates"].to_csv(output / "duplicate_candidates.csv", index=False)

summary = {
    "run_id": run_id,
    "input_records": len(collection["features"]),
    "selection_counts": dict(Counter(result["selection"].selection_status)),
    "selected_records": len(institutions),
    "category_memberships": dict(Counter(c for cats in institutions.categories for c in cats.split(";") if c)),
    "missing_names": int(institutions.missing_name.sum()),
    "identity_issues": int(institutions.identity_issue.sum()),
    "repeated_identity_records": int(institutions.repeated_identity.sum()),
    "boundary_edge_records": int(institutions.on_boundary.sum()),
    "location_representation": dict(Counter(institutions.location_representation)),
    "candidate_pairs": len(result["duplicate_candidates"]),
    "records_in_candidate_pairs": int((institutions.candidate_pair_count > 0).sum()),
    "inclusion_predicate": "boundary.covers(point); include boundary edges, exclude polygon-hole interiors",
    "review_thresholds_m": CONFIG["inventory_audit"],
    "coordinate_review": "All retained points remain not_independently_verified; numeric validity and municipal inclusion are not positional ground truth.",
}
summary_path = ROOT / "metadata/ethekwini_inventory_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")

# Fail before registering outputs if a GIS export changes IDs, counts or CRS.
geojson_check = gpd.read_file(output / "institutions.geojson")
gpkg_check = gpd.read_file(output / "institutions.gpkg")
assert len(geojson_check) == len(gpkg_check) == len(institutions)
assert list(geojson_check.institution_id) == list(institutions.institution_id)
assert gpkg_check.crs.to_string() == CONFIG["analysis_crs"]
assert geojson_check.crs.to_epsg() == 4326 and geojson_check.is_valid.all()
assert all(boundary.to_crs(4326).geometry.union_all().covers(p) for p in geojson_check.geometry)
assert len(result["selection"]) == len(collection["features"])
assert int((result["selection"].selection_status == "selected").sum()) == len(institutions)
print("CSV, GeoJSON and projected GeoPackage written; spatial/count checks passed.")


In [ ]:
run_directory = ROOT / "metadata/runs" / run_id
run_directory.mkdir(parents=True, exist_ok=False)
(run_directory / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
environment = {"python": platform.python_version(), "geopandas": gpd.__version__,
               "pandas": pd.__version__, "shapely": shapely.__version__, "pyproj": pyproj.__version__}
(run_directory / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
code_paths = ["src/heritage_flood/inventory.py", "notebooks/01_inventory_audit.ipynb"]
(run_directory / "code_checksums.json").write_text(json.dumps({
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in code_paths}, indent=2) + "\n")
output_paths = [str((output / name).relative_to(ROOT)) for name in
                ["institutions.geojson", "institutions.gpkg", "institutions.csv",
                 "institution_selection.csv", "duplicate_candidates.csv"]]
output_paths.append(str(summary_path.relative_to(ROOT)))
(run_directory / "checksums.json").write_text(json.dumps({
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in output_paths}, indent=2) + "\n")
source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
input_checksums = {CONFIG["institution_source"]: source_hash,
                   CONFIG["boundary_path"]: expected_boundary_hash}
(run_directory / "input_checksums.json").write_text(json.dumps(input_checksums, indent=2) + "\n")
source_row = {
    "dataset_id": CONFIG["institution_dataset_id"], "title": "South African cultural institutions OSM repository snapshot",
    "provider": "OpenStreetMap contributors; existing project download", "version": "Local snapshot; original acquisition date unknown",
    "source_url_or_asset_id": "https://www.openstreetmap.org/copyright", "status": "accessed",
    "accessed_at_utc": started.isoformat(), "geographic_coverage": "South Africa",
    "resolution_or_scale": "OSM points; ways and relations represented by bounding-box centers",
    "crs": "EPSG:4326 (GeoJSON)", "license": "ODbL 1.0",
    "redistribution_conditions": "Retain OpenStreetMap attribution and comply with ODbL; outputs remain local for now",
    "limitations": "Incomplete institution coverage; source acquisition date unknown; locations and names not independently verified",
    "storage_location": CONFIG["institution_source"], "sha256": source_hash,
}
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip())
run_row = {
    "run_id": run_id, "started_at_utc": started.isoformat(),
    "input_dataset_ids": CONFIG["institution_dataset_id"] + ";" + CONFIG["boundary_dataset_id"],
    "code_commit": commit, "working_tree_dirty": str(dirty).lower(),
    "config_snapshot_path": str((run_directory / "config.json").relative_to(ROOT)),
    "environment_path": str((run_directory / "environment.json").relative_to(ROOT)),
    "method": "Polygon covers selection; numeric coordinate checks; unmerged proximity/name/identity review candidates",
    "output_locations": ";".join(output_paths),
    "output_checksums_path": str((run_directory / "checksums.json").relative_to(ROOT)),
    "validation_status": "checked",
    "notes": "Spatial selection and export checks passed. Human coordinate and duplicate review remain pending. Boundary effective date unresolved. Input and executed-code hashes retained beside configuration.",
}
for filename, key, row in [("datasets.csv", "dataset_id", source_row),
                           ("processing_runs.csv", "run_id", run_row)]:
    register = ROOT / "metadata" / filename
    with register.open(newline="") as handle:
        reader = csv.DictReader(handle)
        fields, rows = reader.fieldnames, list(reader)
    previous = next((r for r in rows if r[key] == row[key]), None)
    if filename == "datasets.csv" and previous and previous["sha256"] != row["sha256"]:
        raise ValueError("Source snapshot changed; assign a new dataset ID before registering it.")
    rows = [r for r in rows if r[key] != row[key]] + [row]
    with register.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
print("Recorded processing run:", run_id)


## Review interpretation

- All coordinates remain unverified independently; geographic range checks only reject malformed locations.
- Ways and relations retain the source downloader’s bounding-box-center limitation.
- Pairs within 30 m are review candidates, including legitimate co-located institutions. Matching normalized names within 250 m and repeated OSM identities are additional review reasons. These configurable heuristics are not calibrated duplicate probabilities.
- Keep missing names blank and retain original IDs. Review candidate pairs in the exported CSV; no automatic merge is permitted.
- Local processed files are Git-ignored. Copy them to persistent storage and retain run checksums. The boundary’s legal date and redistribution limitations continue to apply.

Next: human review of flagged records, then municipal-floodplain data feasibility. Observer gaps and exposure are not assessed here.